# Step 8 — LSTM Deep Learning Model & Scientific Audit
**Project**: Deep Learning-Based Flood Prediction Using Rainfall Data  
**Dataset**: `data/raw/flood_risk_dataset_india.csv` (10,000 spatial observations across India)  
**Objective**: Conduct data-integrity verification of time-series structure, evaluate scientific suitability of LSTM recurrent neural networks on spatial cross-sectional data, detail required hydrological gauge dataset specifications, and benchmark ML baseline performance.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root directory to path
sys.path.append('..')
from src.train_lstm import (
    verify_timeseries_structure,
    explain_required_timeseries_schema,
    build_pytorch_lstm_template,
    run_lstm_scientific_audit
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Time-Series Data Integrity Audit
Audit raw dataset schema to verify whether genuine chronological/time-series structure exists before building recurrent models.

In [2]:
df_raw = pd.read_csv('../data/raw/flood_risk_dataset_india.csv')
audit = verify_timeseries_structure(df_raw)

print('='*60)
print('TIME-SERIES STRUCTURE AUDIT RESULTS')
print('='*60)
for key, val in audit.items():
    print(f"{key:30s}: {val}")
print('='*60)

TIME-SERIES STRUCTURE AUDIT RESULTS
dataset_name                  : flood_risk_dataset_india.csv
total_records                 : 10000
total_columns                 : 14
time_columns_found            : []
has_timestamp                 : False
spatial_coordinates_present   : True
is_time_series_valid          : False
scientific_recommendation     : SPATIAL_TABULAR_DATA_LSTM_INAPPROPRIATE


## 2. Scientific Rationale: Why LSTM Cannot Be Applied to Spatial Data
> **Methodological Determination**:
> 1. **No Timestamp Attribute**: `flood_risk_dataset_india.csv` contains zero date or time columns.
> 2. **Spatial Cross-Section**: The 10,000 observations represent spatial coordinates across India (Latitude 8.0°–37.0°N, Longitude 68.0°–97.0°E).
> 3. **Data Integrity Standard**: Fabricating sliding window sequences over unordered spatial rows violates physical reality and introduces artificial temporal autocorrelation.
> 4. Per strict project rules (*'If the dataset does NOT contain meaningful sequential/time-series data, do NOT fabricate a sequence... Stop the LSTM implementation until a suitable time-series dataset is available'*), **LSTM training is halted on this spatial dataset** to prevent data corruption.

## 3. Required Hydrological Time-Series Dataset Specification
Specifications of the continuous gauge dataset required for valid LSTM recurrent modeling:

In [3]:
schema_spec = explain_required_timeseries_schema()
print('--- Required Time-Series Dataset Specifications ---')
for key, val in schema_spec.items():
    print(f"\n[{key.upper()}]")
    if isinstance(val, list):
        for item in val:
            print(f"  - {item}")
    else:
        print(f"  {val}")

--- Required Time-Series Dataset Specifications ---

[REQUIRED_COLUMNS]
  - Station_ID (Unique identifier per river/meteorological gauge station)
  - Timestamp (Hourly or daily ISO datetime index, e.g. 2026-01-01 00:00:00)
  - Precipitation_mm (Continuous historical rainfall depth)
  - River_Discharge_m3s (Upstream river streamflow rate)
  - Water_Level_m (Gauge water stage height)
  - Soil_Moisture_pct (Continuous soil moisture saturation)
  - Flood_Occurred (Target flag at time t or t+k)

[REQUIRED_TEMPORAL_FREQUENCY]
  Continuous fixed interval (e.g. hourly, 6-hourly, daily) without missing time gaps

[MINIMUM_SEQUENCE_LENGTH]
  Lookback window N = 7 to 30 past timesteps per station

[TRAIN_TEST_SPLIT_RULE]
  Strict chronological split (e.g. Train: 2020-2023, Test: 2024)


## 4. Production-Ready PyTorch / Keras LSTM Architecture Template
Architecture implementation template designed for future chronological datasets:

In [4]:
lstm_model = build_pytorch_lstm_template(input_dim=30, hidden_dim=64, num_layers=2, dropout=0.2)
if lstm_model is not None:
    print('\nInstantiated PyTorch LSTM Architecture:')
    print(lstm_model)


Instantiated PyTorch LSTM Architecture:
FloodLSTM(
  (lstm): LSTM(30, 64, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


## 5. Comparative Performance Summary: Baseline ML vs. LSTM Audit
Load final evaluation comparison table exported to `results/ml_vs_lstm_benchmark.csv`:

In [5]:
audit_dict, bench_df = run_lstm_scientific_audit(data_dir='../data/raw', results_dir='../results')

print('='*70)
print('FINAL MODEL EVALUATION & BENCHMARK SUMMARY TABLE')
print('='*70)
print(bench_df.to_string(index=False))
print('='*70)

FINAL MODEL EVALUATION & BENCHMARK SUMMARY TABLE
               Model           Accuracy          Precision             Recall           F1-score
 Logistic Regression             0.5075             0.5116             0.5668             0.5378
       Decision Tree              0.504             0.5093             0.5153             0.5123
       Random Forest              0.505             0.5098             0.5381             0.5236
             XGBoost              0.517              0.521             0.5519              0.536
LSTM (Deep Learning) N/A (Spatial Data) N/A (Spatial Data) N/A (Spatial Data) N/A (Spatial Data)


## 6. Final Validation Checklist & Baseline Conclusion
- [x] **Time-Series Verification**: Confirmed 0 date/time attributes in `flood_risk_dataset_india.csv`.
- [x] **Data Leakage Prevention**: Scaler fitted strictly on $X_{train}$; test set completely isolated.
- [x] **No Sequence Fabrication**: Artificial temporal sliding windows were NOT applied to spatial rows.
- [x] **Zero Results Fabricated**: Empirical ML evaluation metrics reported strictly from execution.
- [x] **Definitive Baseline Recommendation**: **XGBoost (Accuracy: 0.5170, F1: 0.5360)** and **Logistic Regression (Recall: 0.5668)** are the optimal baseline models for this spatial dataset.